# 🤖 Project: PPE Detection using Convolutional Neural Networks (CNN)
## Goal: Build a classification model from scratch to detect "Helmet" vs "Head".



## 1. Introduction & Environment Setup

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import zipfile
from sklearn.model_selection import train_test_split

# Keras Components
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout, Input, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils import class_weight
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# --- DATASET UNZIPPING ---
dataset_dir = "../dataset"
zip_filename = "../data/Hard Hat Workers.v2-raw.multiclass.zip"

if os.path.exists(zip_filename):
    print(f"ℹ️ Info: '{zip_filename}' found. Checking file properties...")
    file_size = os.path.getsize(zip_filename)
    print(f"ℹ️ File size: {file_size} bytes")
    # Use 'file' command to check the actual file type
    # This command is usually available in Linux environments like Colab
    print(f"ℹ️ Actual file type (from system command):")
    !file --mime-type "$zip_filename"

    try:
        with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
            zip_ref.extractall(dataset_dir)
        print(f"✅ Success: Dataset extracted to '{dataset_dir}'")
    except zipfile.BadZipFile:
        print(f"❌ Error: '{zip_filename}' is not a valid zip file. Please ensure the file is complete and uncorrupted.")
    except Exception as e:
        print(f"❌ An unexpected error occurred during extraction: {e}")
else:
    print(f"⚠️ Warning: {zip_filename} not found. Ensure the zip is in the root folder.")

ℹ️ Info: '../data/Hard Hat Workers.v2-raw.multiclass.zip' found. Checking file properties...
ℹ️ File size: 242613205 bytes
ℹ️ Actual file type (from system command):


'file' n�o � reconhecido como um comando interno
ou externo, um programa oper�vel ou um arquivo em lotes.


✅ Success: Dataset extracted to '../dataset'


## 2. Data Preparation (CSV to Flow)

In [22]:
# Path to the training metadata
train_csv_path = os.path.join(dataset_dir, "train", "_classes.csv")
train_img_dir = os.path.join(dataset_dir, "train")

# Load and clean the CSV
train_df = pd.read_csv(train_csv_path)
train_df.columns = [c.strip() for c in train_df.columns] # Remove white spaces

# Conversion for Binary Classification:
# We need to map the one-hot columns back to a single string label for flow_from_dataframe
def get_label(row):
    if row['helmet'] == 1: return 'helmet'
    return 'head'

train_df['label'] = train_df.apply(get_label, axis=1)

# Display first few rows to confirm
print("--- Training Data Sample ---")
display(train_df[['filename', 'label']].head())

--- Training Data Sample ---


,filename,label
0,003626_jpg.rf.0024e8fc3c8c3f411962ca8dab7b8e92...,helmet
1,004434_jpg.rf.002a70f061745a217db4320ae7b402a7...,head
2,004858_jpg.rf.002ab521984d81c7400faa6f916f5a01...,head
3,002310_jpg.rf.0008cd4590d2edb0e1447329236d9c11...,helmet
4,004785_jpg.rf.002ffb29898b3cba483a76e8b73d91a8...,helmet


## 3. Data Augmentation & Generators

In [25]:
train_df['absolute_path'] = train_df['filename'].apply(
    lambda x: os.path.normpath(os.path.join(os.path.abspath(train_img_dir), x))
)

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

# Validação SEM aumento de dados
val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

# --- PASSO 3: CRIAR OS GERADORES ---
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=None,  # Como temos o caminho completo, directory deve ser None
    x_col="absolute_path",
    y_col="label",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="training",
    shuffle=True
)

val_generator = val_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=None,
    x_col="absolute_path",
    y_col="label",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation",
    shuffle=False
)

Found 4216 validated image filenames belonging to 2 classes.
Found 1053 validated image filenames belonging to 2 classes.


In [26]:
print("Contagem de classes no treino:")
print(train_df['label'].value_counts())

Contagem de classes no treino:
label
helmet    4832
head       437
Name: count, dtype: int64


## 4. CNN Model Architecture (From Scratch)

In [29]:
# --- 1. Load Pre-trained Base ---
# We use MobileNetV2 for its efficiency in real-time detection
base_model = MobileNetV2(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    include_top=False, # We remove the original classification layer
    weights='imagenet'
)
base_model.trainable = False # Freeze layers to keep pre-trained knowledge

# --- 2. Build Custom Classifier ---
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.4), # Increased dropout to force the model to generalize
    Dense(1, activation='sigmoid') # Sigmoid for 'Helmet' vs 'Head'
])

model.compile(
    optimizer=Adam(learning_rate=1e-4), # Lower learning rate to avoid losing base weights
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)

## 5. Training & Evaluation

In [30]:

# Calcula pesos para compensar o desequilíbrio
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weights_dict = dict(enumerate(weights))

# Agora adicione isso no seu model.fit:
history = model.fit(
    train_generator,
    epochs=30,
    validation_data=val_generator,
    class_weight=class_weights_dict, 
    callbacks=[checkpoint, early_stop, reduce_lr]
)

Epoch 1/30
132/132 ━━━━━━━━━━━━━━━━━━━━ 238s 2s/step - accuracy: 0.7289 - loss: 0.5638 - precision_2: 0.9663 - recall_2: 0.7289 - val_accuracy: 0.8870 - val_loss: 0.2774 - val_precision_2: 0.9842 - val_recall_2: 0.8925 - learning_rate: 1.0000e-04
Epoch 2/30
132/132 ━━━━━━━━━━━━━━━━━━━━ 105s 514ms/step - accuracy: 0.8446 - loss: 0.3896 - precision_2: 0.9834 - recall_2: 0.8444 - val_accuracy: 0.9145 - val_loss: 0.2256 - val_precision_2: 0.9901 - val_recall_2: 0.9171 - learning_rate: 1.0000e-04
Epoch 3/30
132/132 ━━━━━━━━━━━━━━━━━━━━ 66s 497ms/step - accuracy: 0.8494 - loss: 0.3370 - precision_2: 0.9867 - recall_2: 0.8467 - val_accuracy: 0.9345 - val_loss: 0.1720 - val_precision_2: 0.9861 - val_recall_2: 0.9427 - learning_rate: 1.0000e-04
Epoch 4/30
132/132 ━━━━━━━━━━━━━━━━━━━━ 65s 487ms/step - accuracy: 0.8733 - loss: 0.3001 - precision_2: 0.9860 - recall_2: 0.8739 - val_accuracy: 0.9402 - val_loss: 0.1649 - val_precision_2: 0.9862 - val_recall_2: 0.9488 - learning_rate: 2.0000e-05
Epoch

## 6. Deployment-Ready Artifacts

In [31]:
# To download from Colab
try:
    from google.colab import files
    files.download('../models/best_epi_model.keras')
except:
    print("Running locally. Model saved in the root directory.")

Running locally. Model saved in the root directory.
